# Raw categorical feature and leakage audit

This exploratory notebook reads the synthetic raw transaction data in chunks. It inventories categorical fields, compares every observed value between normal and laundering transactions, and reports effect-size screening statistics. It does not change the modelling feature contract or train a model.

A strong class association is a reason to inspect simulator provenance and time availability; it is not, by itself, proof that a field is leakage. Direct scenario, target and simulator-generation fields are marked as never-use inputs for review.

## Outputs

The audit writes CSV/JSON tables to `results/insights/feature_audit` and independent PNG figures to `results/images/09_context_diagnostics/feature_audit`. High-cardinality fields receive a complete cardinality summary and a capped top-value table rather than an impractically large display.

In [ ]:
"""Audit raw categorical fields for class-conditional differences and shortcuts.

This is an exploratory audit only.  It does not alter the modelling contract,
fit a model, or make a leakage decision from a p-value.  The raw CSV is read
in chunks so the audit remains usable for the full synthetic dataset.
"""

from __future__ import annotations

from collections import Counter, defaultdict
import json
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency


if "__file__" in globals():
    ROOT = Path(__file__).resolve().parents[1]
else:
    # Notebook execution normally starts from the project root.  The fallback
    # also supports launching the notebook from its own ``notebooks`` folder.
    ROOT = Path.cwd()
    if not (ROOT / "data" / "raw").exists() and (ROOT.parent / "data" / "raw").exists():
        ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.sa_aml_simulation.schema import PUBLIC_FINAL_COLUMNS
RAW_PATH = ROOT / "results" / "simulated_data" / "SA_SAML_D.csv"
FINAL_PATH = ROOT / "results" / "simulated_data" / "SA_SAML_D.csv"
TABLE_ROOT = ROOT / "results" / "insights" / "feature_audit"
FIGURE_ROOT = ROOT / "results" / "images" / "09_context_diagnostics" / "feature_audit"
TARGET = "Is_laundering"
CHUNK_ROWS = 100_000
SAMPLE_ROWS = 20_000
MAX_DETAILED_VALUES = 500
TOP_VALUES_PER_FIGURE = 20
MIN_SUPPORT_FOR_EFFECT = 20

# Fields supplied as explicit simulator-risk/context candidates.  They are
# shown in the audit and marked for review; this notebook does not promote
# them automatically into the primary model.
NEVER_USE = set("""Is_laundering Account_is_laundering Laundering_type
AML_scenario_category AML_scenario_code AML_predicate_offence AML_stage AML_risk_sector
AML_scenario_source_keys Scenario_reference_key Scenario_detail Scenario_label
Suspicion_indicator Behaviour_component Typology_stage Legitimate_behaviour
Normal_behavior_category Normal_behavior_source_keys Opened_for_laundering_flag
Scenario_forced_coverage_flag Account_laundering_probability Transaction_purpose_category
First_observed_transaction_date Opened_during_observation_flag Active_observation_days
Account_activity_multiplier Expected_transaction_count Generated_transaction_count
KYC_strength Beneficial_ownership_opacity Age_band Income_band Residency_status Migrant_worker_flag
Cash_intensity_score Cash_intensity_band High_risk_sector_flag High_risk_person_flag
Debit_card_indicator Business_registered Has_trading_permit VAT_registered
Uses_personal_account_for_business Business_premises_type Estimated_daily_cash_turnover
Business_age_years Keeps_business_records Has_bank_loan_access""".split())

IDENTIFIERS = {"Transaction_id", "Account_id", "Sender_account", "Receiver_account", "Counterparty_account"}
REVIEW_CONTEXT = set("""Public_sector_exposure Procurement_vendor Is_PEP PEP_category
Intermediary_account_flag Virtual_asset_exposure Connected_party_count Adverse_media_flag
Targeted_financial_sanctions_match Profile_jurisdiction_risk_tier Typical_foreign_counterparty_country
CTR_threshold_near_miss Round_amount_flag Public_sector_counterparty PEP_related_transaction
High_risk_sector_category High_risk_person_type Counterparty_type Cash_intensity_score
Cash_intensity_band Time Date Direction Amount Payment_currency Received_currency
Sender_bank_location Receiver_bank_location Counterparty_country Payment_type Social_grant_type
Cross_border_transaction SADC_corridor_transaction Jurisdiction_risk_tier Customer_type
Employment_status Employment_sector Income_band Monthly_income_estimate Province Grant_linked
Gender Age Age_band Education_level Nationality_country Residency_status Migrant_worker_flag
Urban_rural Township_or_informal_settlement_flag Financial_inclusion_segment Channel_preference
Primary_social_grant_type Business_registration_status Uses_personal_account_for_business
Estimated_daily_cash_turnover Business_age_years Has_bank_loan_access Informal_business_flag
Business_registered Keeps_business_records Business_premises_type Has_trading_permit VAT_registered
Account_open_date""".split())

# Numeric flags are explicitly included because pandas may infer them as
# integers rather than categorical values.
KNOWN_CATEGORICAL = set("""Direction Payment_currency Received_currency Sender_bank_location
Receiver_bank_location Counterparty_country Payment_type Social_grant_type Counterparty_type
Jurisdiction_risk_tier Customer_type Employment_status Employment_sector Income_band Province
Gender Age_band Education_level Nationality_country Residency_status Urban_rural
Financial_inclusion_segment Channel_preference Primary_social_grant_type Business_registration_status
PEP_category Profile_jurisdiction_risk_tier Typical_foreign_counterparty_country""".split())
KNOWN_CATEGORICAL |= {"Cross_border_transaction", "SADC_corridor_transaction", "Grant_linked",
                      "Migrant_worker_flag", "Township_or_informal_settlement_flag",
                      "Uses_personal_account_for_business", "Has_bank_loan_access",
                      "Informal_business_flag", "Business_registered", "Keeps_business_records",
                      "Has_trading_permit", "VAT_registered", "Procurement_vendor", "Is_PEP",
                      "Intermediary_account_flag", "Virtual_asset_exposure", "Adverse_media_flag",
                      "Targeted_financial_sanctions_match", "CTR_threshold_near_miss",
                      "Round_amount_flag", "Public_sector_counterparty", "PEP_related_transaction",
                      "High_risk_sector_flag", "High_risk_person_flag"}


def policy_group(feature: str) -> str:
    if feature == TARGET or feature in NEVER_USE:
        return "never_use_as_predictive_input"
    if feature in IDENTIFIERS:
        return "identifier_grouping_only"
    if feature in REVIEW_CONTEXT:
        return "candidate_context_or_behaviour_review"
    return "unclassified_review"


def infer_categorical_columns(sample: pd.DataFrame) -> list[str]:
    columns = []
    for feature in sample.columns:
        if feature == TARGET:
            continue
        series = sample[feature]
        numeric = pd.api.types.is_numeric_dtype(series)
        low_cardinality_numeric = numeric and series.nunique(dropna=False) <= 20
        if feature in KNOWN_CATEGORICAL or not numeric or low_cardinality_numeric:
            columns.append(feature)
    return columns


def _string_values(series: pd.Series) -> pd.Series:
    return series.astype("string").fillna("<MISSING>")


def audit_values(columns: list[str]) -> tuple[dict[str, Counter], dict[str, dict[str, Counter]], int, dict[str, int]]:
    totals: dict[str, Counter] = {feature: Counter() for feature in columns}
    by_class: dict[str, dict[str, Counter]] = {
        feature: {"0": Counter(), "1": Counter()} for feature in columns
    }
    class_counts = {"0": 0, "1": 0}
    rows = 0
    usecols = [TARGET, *columns]
    for chunk in pd.read_csv(RAW_PATH, usecols=usecols, chunksize=CHUNK_ROWS, low_memory=False):
        target = pd.to_numeric(chunk[TARGET], errors="coerce").fillna(0).astype("int8")
        class_counts["0"] += int((target == 0).sum())
        class_counts["1"] += int((target == 1).sum())
        rows += len(chunk)
        for feature in columns:
            values = _string_values(chunk[feature])
            totals[feature].update(values.tolist())
            for label in (0, 1):
                selected = values.loc[target.eq(label)]
                by_class[feature][str(label)].update(selected.tolist())
    return totals, by_class, rows, class_counts


def _cramers_v(table: np.ndarray) -> tuple[float, float]:
    if table.shape[0] < 2 or table.sum() == 0:
        return 0.0, 1.0
    chi2, p_value, _, _ = chi2_contingency(table, correction=False)
    n = table.sum()
    phi2 = chi2 / n
    rows, cols = table.shape
    denominator = max(min(rows - 1, cols - 1), 1)
    return float(np.sqrt(phi2 / denominator)), float(p_value)


def build_tables(columns, totals, by_class, rows, class_counts, sample) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    overall_positive_rate = class_counts["1"] / max(rows, 1)
    summary_rows, value_rows = [], []
    for feature in columns:
        values = totals[feature]
        ordered = sorted(values.items(), key=lambda item: (-item[1], str(item[0])))
        matrix = np.array([[by_class[feature]["0"].get(value, 0), by_class[feature]["1"].get(value, 0)] for value, _ in ordered], dtype=float)
        cramers, p_value = _cramers_v(matrix)
        supported = [(value, count) for value, count in ordered if count >= MIN_SUPPORT_FOR_EFFECT]
        rates = []
        for value, count in supported:
            positives = by_class[feature]["1"].get(value, 0)
            rates.append((value, positives / count, count))
        strongest = max(rates, key=lambda item: item[1], default=("", np.nan, 0))
        weakest = min(rates, key=lambda item: item[1], default=("", np.nan, 0))
        summary_rows.append({
            "feature": feature, "policy_group": policy_group(feature), "sample_dtype": str(sample[feature].dtype),
            "unique_values_full": len(values), "missing_count_full": values.get("<MISSING>", 0),
            "overall_positive_rate": overall_positive_rate, "cramers_v": cramers, "chi_square_p_value": p_value,
            "min_supported_category_positive_rate": weakest[1], "min_supported_category": weakest[0],
            "max_supported_category_positive_rate": strongest[1], "max_supported_category": strongest[0],
            "max_category_rate_difference": (strongest[1] - weakest[1]) if rates else np.nan,
            "max_category_lift": (strongest[1] / overall_positive_rate) if rates and overall_positive_rate else np.nan,
        })
        detailed = ordered if len(ordered) <= MAX_DETAILED_VALUES else ordered[:MAX_DETAILED_VALUES]
        reported_top = len(ordered) > MAX_DETAILED_VALUES
        for value, count in detailed:
            normal = by_class[feature]["0"].get(value, 0)
            laundering = by_class[feature]["1"].get(value, 0)
            total = normal + laundering
            value_rows.append({
                "feature": feature, "policy_group": policy_group(feature), "value": value,
                "normal_count": normal, "laundering_count": laundering, "total_count": total,
                "normal_share_within_class": normal / max(class_counts["0"], 1),
                "laundering_share_within_class": laundering / max(class_counts["1"], 1),
                "category_positive_rate": laundering / max(total, 1),
                "category_lift_vs_overall": (laundering / max(total, 1)) / overall_positive_rate if overall_positive_rate else np.nan,
                "all_values_reported": not reported_top,
            })
    return pd.DataFrame(summary_rows), pd.DataFrame(value_rows), pd.DataFrame([
        {"class": "Normal (0)", "count": class_counts["0"], "share": class_counts["0"] / max(rows, 1)},
        {"class": "Laundering (1)", "count": class_counts["1"], "share": class_counts["1"] / max(rows, 1)},
    ])


def _safe_name(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9_-]+", "_", value).strip("_").lower()[:90]


def _save(figure, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    figure.tight_layout()
    figure.savefig(path, dpi=260, bbox_inches="tight")
    plt.close(figure)


def make_figures(summary: pd.DataFrame, values: pd.DataFrame, class_counts: pd.DataFrame) -> None:
    FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
    figure, axis = plt.subplots(figsize=(7.2, 4.8))
    bars = axis.bar(class_counts["class"], class_counts["count"], color=["#0072B2", "#D55E00"])
    axis.set_yscale("log")
    axis.set(ylabel="Transactions (log scale)", title="Raw transaction class imbalance")
    for bar, count in zip(bars, class_counts["count"]):
        axis.text(bar.get_x() + bar.get_width() / 2, count, f"{int(count):,}", ha="center", va="bottom")
    _save(figure, FIGURE_ROOT / "01_raw_class_imbalance.png")

    for column, title, filename in [("unique_values_full", "Categorical cardinality in raw data", "02_categorical_cardinality.png"),
                                    ("cramers_v", "Class association by categorical feature (Cramer's V)", "03_categorical_cramers_v.png"),
                                    ("max_category_rate_difference", "Largest supported category positive-rate difference", "04_category_positive_rate_difference.png")]:
        ordered = summary.sort_values(column).tail(30)
        figure, axis = plt.subplots(figsize=(9.2, max(5.0, len(ordered) * .25)))
        axis.barh(ordered["feature"], ordered[column], color="#0072B2")
        axis.set(xlabel=column.replace("_", " ").title(), title=title)
        axis.grid(axis="x", alpha=.25)
        _save(figure, FIGURE_ROOT / filename)

    per_feature = FIGURE_ROOT / "categorical_distributions"
    for feature in summary["feature"]:
        table = values.loc[values["feature"].eq(feature)].sort_values("total_count", ascending=False).head(TOP_VALUES_PER_FIGURE).copy()
        if table.empty:
            continue
        table = table.sort_values("laundering_share_within_class")
        y = np.arange(len(table))
        figure, axis = plt.subplots(figsize=(9.5, max(4.5, len(table) * .30)))
        axis.barh(y + .18, table["normal_share_within_class"], height=.34, label="Normal", color="#0072B2")
        axis.barh(y - .18, table["laundering_share_within_class"], height=.34, label="Laundering", color="#D55E00")
        axis.set(yticks=y, yticklabels=table["value"].astype(str), xlabel="Share within class", title=f"{feature}: class-conditional top-value distribution")
        axis.grid(axis="x", alpha=.25)
        axis.legend()
        _save(figure, per_feature / f"{_safe_name(feature)}.png")


def main() -> None:
    if not RAW_PATH.exists():
        raise FileNotFoundError(RAW_PATH)
    sample = pd.read_csv(RAW_PATH, nrows=SAMPLE_ROWS, low_memory=False)
    columns = infer_categorical_columns(sample)
    totals, by_class, rows, class_counts = audit_values(columns)
    summary, values, class_table = build_tables(columns, totals, by_class, rows, class_counts, sample)
    TABLE_ROOT.mkdir(parents=True, exist_ok=True)
    summary.sort_values(["policy_group", "feature"]).to_csv(TABLE_ROOT / "categorical_feature_leakage_screen.csv", index=False)
    values.sort_values(["feature", "total_count"], ascending=[True, False]).to_csv(TABLE_ROOT / "categorical_value_class_comparison.csv", index=False)
    class_table.to_csv(TABLE_ROOT / "raw_class_counts.csv", index=False)
    schema = pd.DataFrame({"feature": sample.columns, "sample_dtype": [str(sample[col].dtype) for col in sample.columns],
                           "sample_unique_values": [int(sample[col].nunique(dropna=False)) for col in sample.columns],
                           "sample_missing": [int(sample[col].isna().sum()) for col in sample.columns],
                           "inferred_categorical": [col in columns for col in sample.columns],
                           "policy_group": [policy_group(col) for col in sample.columns]})
    schema.to_csv(TABLE_ROOT / "raw_schema_inventory.csv", index=False)
    final_header = pd.read_csv(FINAL_PATH, nrows=0)
    final_columns = list(final_header.columns)
    missing_final_columns = [column for column in PUBLIC_FINAL_COLUMNS if column not in final_columns]
    unexpected_final_columns = [column for column in final_columns if column not in PUBLIC_FINAL_COLUMNS]
    if missing_final_columns or unexpected_final_columns or final_columns != list(PUBLIC_FINAL_COLUMNS):
        raise RuntimeError(
            f"Final schema mismatch. Missing={missing_final_columns}; unexpected={unexpected_final_columns}"
        )
    final_sample = pd.read_csv(FINAL_PATH, nrows=SAMPLE_ROWS, low_memory=False)
    final_categorical = infer_categorical_columns(final_sample)
    final_schema = pd.DataFrame({
        "column_order": range(1, len(final_columns) + 1),
        "feature": final_columns,
        "sample_dtype": [str(final_sample[column].dtype) for column in final_columns],
        "audit_view": ["target" if column == TARGET else ("categorical_screen" if column in final_categorical else "numeric_or_identifier") for column in final_columns],
        "policy_group": [policy_group(column) for column in final_columns],
    })
    final_schema.to_csv(TABLE_ROOT / "final_schema_inventory.csv", index=False)
    make_figures(summary, values, class_table)
    manifest = {"raw_path": str(RAW_PATH), "final_path": str(FINAL_PATH), "rows_audited": rows, "categorical_features_audited": columns,
                "raw_column_count": int(len(sample.columns)), "categorical_feature_count": len(columns),
                "final_column_count": len(final_columns), "final_categorical_column_count": len(final_categorical),
                "final_numeric_or_identifier_column_count": len(final_columns) - len(final_categorical) - 1,
                "final_columns": final_columns, "chunk_rows": CHUNK_ROWS, "sample_rows_for_dtype_inference": SAMPLE_ROWS,
                "target": TARGET, "note": "Association statistics are screening evidence, not automatic leakage decisions. Inspect simulator provenance and temporal availability before modelling."}
    (TABLE_ROOT / "audit_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print(json.dumps({"rows_audited": rows, "raw_columns": len(sample.columns), "final_columns": len(final_columns), "final_categorical_columns": len(final_categorical), "final_numeric_or_identifier_columns": len(final_columns) - len(final_categorical) - 1, "normal": class_counts["0"], "laundering": class_counts["1"], "output": str(TABLE_ROOT)}, indent=2))


if __name__ == "__main__":
    main()
